# MELD EFR — training del progetto da Google Drive

Questo notebook **non contiene il training code duplicato**: usa direttamente il package Python presente nella cartella del progetto su Google Drive. In questo modo il codice usato su Colab è lo stesso che verrà poi pushato su GitHub e usato da terminale per l'inference.

In [5]:
from google.colab import drive
drive.mount('/content/drive',  force_remount=True)

Mounted at /content/drive


## 1. Configurazione percorsi

Modifica `PROJECT_DIR` in modo che punti alla cartella `meld-efr-cli` caricata sul tuo Drive. I pesi e le metriche verranno salvati dentro `PROJECT_DIR/artifacts`.

In [7]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/meld-efr')
TRAIN_JSON = PROJECT_DIR / 'data' / 'MELD_train_efr.json'
VAL_JSON = PROJECT_DIR / 'data' / 'MELD_val_efr.json'
TEST_JSON = PROJECT_DIR / 'data' / 'MELD_test_efr.json'
OUTPUT_DIR = PROJECT_DIR / 'artifacts'

assert PROJECT_DIR.exists(), f'Cartella progetto non trovata: {PROJECT_DIR}'
assert TRAIN_JSON.exists(), f'Train JSON non trovato: {TRAIN_JSON}'
assert VAL_JSON.exists(), f'Validation JSON non trovato: {VAL_JSON}'
assert TEST_JSON.exists(), f'Test JSON non trovato: {TEST_JSON}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_DIR)
print('Artifacts:', OUTPUT_DIR)

Project: /content/drive/MyDrive/meld-efr
Artifacts: /content/drive/MyDrive/meld-efr/artifacts


## 2. Installazione del progetto

L'installazione editable fa sì che Colab importi direttamente il codice nella cartella su Drive.

In [8]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_DIR)], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '/content/drive/MyDrive/meld-efr'], returncode=0)

## 3. Training

Il best checkpoint viene scritto direttamente su Google Drive in `artifacts/best_model.pt`. Puoi cambiare gli iperparametri qui senza modificare il package.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'meld_efr', 'train',
    '--train-path', str(TRAIN_JSON),
    '--val-path', str(VAL_JSON),
    '--test-path', str(TEST_JSON),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '8',
    '--batch-size', '4',
    '--grad-accum-steps', '2',
    '--encoder-lr', '2e-5',
    '--head-lr', '3e-4',
    '--patience', '3',
    '--device', 'auto',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(PROJECT_DIR))

## 4. Verifica artefatti e metriche

In [ ]:
import json
from pathlib import Path

for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'{path.name:24s} {path.stat().st_size / (1024**2):8.2f} MB')

with (OUTPUT_DIR / 'metrics.json').open('r', encoding='utf-8') as f:
    metrics = json.load(f)
metrics

## 5. Prediction di prova con il checkpoint salvato

Questa è la stessa CLI che funzionerà dopo `git clone`.

In [ ]:
EXAMPLE_JSON = PROJECT_DIR / 'examples' / 'dialogue.json'
PREDICTIONS_JSON = OUTPUT_DIR / 'example_predictions.json'

cmd = [
    sys.executable, '-m', 'meld_efr', 'predict',
    '--checkpoint', str(OUTPUT_DIR / 'best_model.pt'),
    '--input', str(EXAMPLE_JSON),
    '--output', str(PREDICTIONS_JSON),
    '--device', 'auto',
]
subprocess.run(cmd, check=True, cwd=str(PROJECT_DIR))

## 6. Dopo il training

La cartella `PROJECT_DIR` contiene già codice + `artifacts/best_model.pt`. Puoi quindi copiarla/scaricarla e versionarla. Per checkpoint grandi, usa Git LFS come indicato nel `README.md` del progetto.